In [3]:
# -*- coding: utf-8 -*-
from Library import utils, dataset
import os
from tensorflow import keras
from tqdm import tqdm
import numpy as np
from datetime import datetime
import logging
import matplotlib.pyplot as plt

# ===================================================================
# FUNGSI BANTUAN
# ===================================================================
def parse_prediction(pred_val):
    """Menormalisasi luaran inferensi menjadi Ground Truth integer (1=LE, 0=Noise)"""
    if isinstance(pred_val, str):
        return 1 if pred_val.lower() == 'le' else 0
    return int(pred_val)

def plot_comprehensive_metrics(metrics, save_path):
    """Membuat diagram batang metrik komprehensif dengan penanganan kunci yang aman"""
    labels = ['Accuracy', 'Recall', 'Precision', 'FPR', 'F1-Score']
    
    # Menggunakan kunci lowercase untuk menghindari ketidakcocokan (KeyError)
    keys = [
        'accuracy (avg.)',
        'true positive rate (avg.)',
        'positive predictive value (avg.)',
        'false positive rate (avg.)',
        'f1-score (avg.)'
    ]

    # Mengambil nilai secara aman (default 0.0 jika kunci tidak ditemukan)
    values = [metrics.get(k, 0.0) for k in keys]
    print(f"[DEBUG] Nilai yang akan diplot: {values}")

    plt.figure(figsize=(12, 6))
    bars = plt.bar(labels, values, color=['#2c3e50', '#3498db', '#e67e22', '#e74c3c', '#27ae60'])

    plt.ylim(0, 1.1) # Diberi ruang sedikit di atas 1.0 agar teks tidak terpotong
    plt.ylabel('Score')
    plt.title('Performance Metrics Evaluation (Z Component)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"[INFO] Grafik metrik komprehensif tersimpan di: {save_path}")


# ===================================================================
# MAIN EXECUTION
# ===================================================================
if __name__ == "__main__":

    # 1. KONFIGURASI PATH STEAD 5000 Z
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_3C_20260719_062844.json'
    DATA_TAG = "STEAD_5000_unseen"
    MODEL_TAG = "baseline"
    INPUT_WIN = 7
    SAMPLING_RATE = 100

    true_labels = ["NO", "LE"]

    # Path Model dan Embedding
    BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo"
    MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
    
    # PERBAIKAN: Menghapus nama file dari path direktori agar tidak dibaca ganda
    EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

    # 2. PREPARE FILES
    print(f"[INFO] Memuat Dataset STEAD 5000: {KEY_TEST_FILE}")
    test_data = dataset.load_json_data(os.path.join(KEY_DATA_DIR, KEY_TEST_FILE))

    SAVE_BASE = '/Volumes/Extreme SSD/stream_stead/data_stead/benchmark_stead_5000_z'
    now = datetime.now()
    time_str = now.strftime("%d%H%M%S")
    save_dir = os.path.join(SAVE_BASE, f"{MODEL_TAG}_{DATA_TAG}_{time_str}")
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # Logger
    log_file_path = os.path.join(save_dir, "task_log_stead_5000_z.txt")
    logging.basicConfig(filename=log_file_path, level=logging.INFO, filemode='w',
                        format='%(asctime)s - [%(levelname)s]: %(message)s')
    logger = logging.getLogger()
    logger.addHandler(logging.StreamHandler())

    # Load Model (compile=False ditambahkan untuk menekan warning graf) & embedding Z
    embedding_model = keras.models.load_model(filepath=MODEL_PATH, compile=False)
    embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    embedding_Z_PDFs = utils.embedding_PDFs_1D(embedding_Z, source_list=['noise', 'le'])

    logger.info("Estimating STEAD 5000 Z embedding statistics (KDE) using UUSS reference...")

    # 3. EVALUATION Z
    total_true_Z, total_pred_Z = [], []
    num_points = int(INPUT_WIN * SAMPLING_RATE)
    keys_list = list(test_data.keys())

    logger.info(f"Memproses {len(keys_list)} data STEAD 5000 (komponen Z)...")

    for i in tqdm(range(len(keys_list)), desc="Inference STEAD 5000 Z"):
        record_key = keys_list[i]
        record = test_data[record_key]

        try:
            # Sinyal dan noise komponen Z saja
            Z_n = record["Z_noise"][-num_points:]
            Z_s = record["Z"][:num_points]

            # Ekstraksi embedding komponen Z
            emb_n_Z = utils.latent_codes_1D(Z_n, embedding_model)
            emb_s_Z = utils.latent_codes_1D(Z_s, embedding_model)

            # Inferensi 1C KDE - Noise dan Earthquake (Signal)
            p_n_Z, _, _ = utils.infer_1C_PDFs(emb_n_Z, embedding_Z_PDFs, "Kernel")
            p_s_Z, _, _ = utils.infer_1C_PDFs(emb_s_Z, embedding_Z_PDFs, "Kernel")

            # PERBAIKAN: Ekstraksi prediksi menggunakan fungsi bantuan yang aman
            total_true_Z.extend([0, 1])
            total_pred_Z.extend([
                parse_prediction(p_n_Z),
                parse_prediction(p_s_Z)
            ])

        except Exception as e:
            logger.error(f"Error pada {record_key}: {e}")
            continue

    # 4. METRICS & SAVE
    matrix_Z, metrics_Z = utils.calc_confusion_metrics(total_true_Z, total_pred_Z)

    # Visualisasi Confusion Matrix
    fig_Z = utils.plot_confusion(f"{MODEL_TAG} {DATA_TAG} Z", true_labels, matrix_Z, metrics_Z)
    fig_Z.savefig(os.path.join(save_dir, "STEAD_5000_Z_confusion.jpg"), dpi=300)
    plt.close(fig_Z)

    # Simpan metrics ke JSON
    dataset.save_json_data(os.path.join(save_dir, "stead_5000_Z_metrics.json"), metrics_Z)

    # Plot comprehensive metrics (bar chart)
    plot_comprehensive_metrics(metrics_Z, os.path.join(save_dir, "stead_5000_Z_metrics_bar.jpg"))

    # 5. HASIL AKHIR
    logger.info("\n" + "="*40)
    # Memastikan penarikan data menggunakan fallback (0) jika None dikembalikan
    logger.info(f"STEAD 5000 Z ACCURACY: {metrics_Z.get('accuracy (avg.)', 0)}")
    logger.info(f"STEAD 5000 Z F1-SCORE: {metrics_Z.get('f1-score (avg.)', 0)}")
    logger.info("="*40)
    logger.info("Pengujian STEAD_5000_Z Selesai.")

[INFO] Memuat Dataset STEAD 5000: STEAD_5000_3C_20260719_062844.json


Estimating STEAD 5000 Z embedding statistics (KDE) using UUSS reference...
Estimating STEAD 5000 Z embedding statistics (KDE) using UUSS reference...
Memproses 5000 data STEAD 5000 (komponen Z)...
Memproses 5000 data STEAD 5000 (komponen Z)...
Inference STEAD 5000 Z: 100%|██████████| 5000/5000 [00:38<00:00, 130.06it/s]


STEAD 5000 Z ACCURACY: 0
STEAD 5000 Z ACCURACY: 0
STEAD 5000 Z F1-SCORE: 0
STEAD 5000 Z F1-SCORE: 0
Pengujian STEAD_5000_Z Selesai.
Pengujian STEAD_5000_Z Selesai.


Accuracy: [0.7054, 0.7054]
Accuracy (avg.): 0.7054
True positive rate: [0.512, 0.8988]
True positive rate (avg.): 0.7054
True negative rate: [0.8988, 0.512]
True negative rate (avg.): 0.7054
Positive predictive value: [0.8349641226353555, 0.6481107585809057]
Positive predictive value (avg.): 0.7415374406081306
Negative predictive value: [0.6481107585809057, 0.8349641226353555]
Negative predictive value (avg.): 0.7415374406081306
False positive rate: [0.1012, 0.488]
False positive rate (avg.): 0.2946
False negative rate: [0.488, 0.1012]
False negative rate (avg.): 0.2946
False discovery rate: [0.16503587736464448, 0.35188924141909433]
False discovery rate (avg.): 0.2584625593918694
F1-score: [0.634763203570543, 0.7531422825540472]
F1-score (avg.): 0.6939527430622952
[DEBUG] Nilai yang akan diplot: [0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] Grafik metrik komprehensif tersimpan di: /Volumes/Extreme SSD/stream_stead/data_stead/benchmark_stead_5000_z/baseline_STEAD_5000_unseen_08071152/stead_5000_Z_m